# RAG 도구 및 웹 검색 기능

웹 검색을 통해 데이터를 수집하고, 수집한 문서를 Chroma 벡터 데이터베이스에 저장하거나 검색하는 기능을 정의합니다.  
LangChain 기반 RAG(Retrieval-Augmented Generation) 시스템에서 사용하는 문서 처리 함수와 에이전트가 호출할 수 있는 Tool을 제공합니다.

## 기능
| 함수 | 설명 | Agent Tool |
|---|---|:---:|
| `load_web_page()` | 웹페이지에서 텍스트를 추출하고 불필요한 공백을 정리합니다. | ❌ |
| `web_search()` | Tavily API로 웹을 검색하고 결과를 JSON 파일로 저장합니다. | ✅ |
| `web_page_to_document()` | 웹 검색 결과를 LangChain Document 객체로 변환합니다. | ❌ |
| `web_page_json_to_documents()` | JSON 파일의 웹 검색 결과를 Document 목록으로 변환합니다. | ❌ |
| `split_documents()` | 긴 문서를 지정한 크기의 청크(Chunk)로 분할합니다. | ❌ |
| `documents_to_chroma()` | 새로운 문서를 임베딩하여 Chroma 벡터 데이터베이스에 저장합니다. | ❌ |
| `add_web_pages_json_to_chroma()` | JSON 파일의 문서를 변환하고 Chroma DB에 저장하는 과정을 수행합니다. | ❌ |
| `retrieve()` | Chroma 벡터 데이터베이스에서 검색어와 유사한 문서를 검색합니다. | ✅ |

In [1]:
import os

# Notebook의 현재 작업 디렉터리
current_path = os.getcwd()

# 데이터 저장 디렉터리 생성
data_directory = os.path.join(current_path, "data")
os.makedirs(data_directory, exist_ok=True)

In [2]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# OpenAI Embedding 설정
embedding = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

# Chroma DB 저장 경로 설정
persist_directory = os.path.join(
    current_path, "data", "chroma_store"
)

# Chroma 객체 생성
vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding
)

### load_web_page() — 웹페이지 텍스트 추출

웹페이지 URL을 입력받아 HTML 문서의 텍스트를 추출하고, 불필요하게 반복되는 줄바꿈과 탭 문자를 정리합니다.  
LangChain의 `WebBaseLoader`를 사용하여 웹페이지를 로드합니다.

**매개변수**
| 매개변수 | 타입 | 설명 |
|---|---|---|
| `url` | `str` | 텍스트를 추출할 웹페이지의 URL |

**반환값**
- `str`: 웹페이지에서 추출하고 정리한 텍스트

**주요 처리 과정**
1. `WebBaseLoader`를 생성하고 지정된 URL의 웹페이지를 로드합니다.
2. 로드된 첫 번째 `Document` 객체에서 `page_content`를 추출합니다.
3. `strip()`으로 텍스트 앞뒤의 공백을 제거합니다.
4. 연속된 줄바꿈(`\n\n\n`)과 탭(`\t\t\t`)을 정리합니다.
5. 정리된 텍스트를 반환합니다.

In [3]:
from langchain_community.document_loaders import WebBaseLoader

def load_web_page(url: str):
    loader = WebBaseLoader(url, verify_ssl=False)

    content = loader.load()
    raw_content = content[0].page_content.strip()

    while '\n\n\n' in raw_content or '\t\t\t' in raw_content:
        raw_content = raw_content.replace('\n\n\n', '\n\n')
        raw_content = raw_content.replace('\t\t\t', '\t\t')

    return raw_content

C:\Users\Unreon\AppData\Local\Temp\ipykernel_24336\3082334981.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


### web_search() — Tavily 웹 검색 Tool
Tavily API를 이용하여 검색어와 관련된 웹페이지를 검색하고, 검색 결과를 JSON 파일로 저장합니다.  
검색 결과에 원본 콘텐츠가 없는 경우 `load_web_page()`를 호출하여 웹페이지 본문을 추가로 수집합니다.  
이 함수는 `@tool` 데코레이터를 사용하여 **LangChain Tool**로 정의됩니다.

**매개변수**
| 매개변수 | 타입 | 설명 |
|---|---|---|
| `query` | `str` | Tavily에서 검색할 검색어 |

**Tavily 검색 옵션**
| 매개변수 | 값 | 설명 |
|---|---|---|
| `search_depth` | `"advanced"` | 고급 검색 모드 사용 |
| `include_raw_content` | `True` | 가능한 경우 웹페이지의 원본 콘텐츠 포함 |

**반환값**
- `results`: 웹 검색 결과 목록
- `resources_json_path`: 검색 결과가 저장된 JSON 파일 경로

**주요 처리 과정**

1. `TavilyClient` 객체를 생성합니다.
2. 검색어를 이용해 Tavily API에 웹 검색을 요청합니다.
3. 검색 결과에서 `raw_content`가 없는 웹페이지를 확인합니다.
4. 필요한 경우 `load_web_page()`로 본문을 수집합니다.
5. 검색 결과를 현재 날짜와 시간이 포함된 JSON 파일로 저장합니다.
6. 검색 결과와 JSON 파일 경로를 반환합니다.

In [4]:
from tavily import TavilyClient
from langchain_core.tools import tool
from datetime import datetime
import json

@tool
def web_search(query: str):
    """
    주어진 query에 대해 웹검색을 하고, 결과를 반환한다.

    Args:
        query (str): 검색어

    Returns:
        dict: 검색 결과
    """
    client = TavilyClient()

    content = client.search(
        query,
        search_depth="advanced",
        include_raw_content=True,
    )

    results = content["results"]

    for result in results:
        if result["raw_content"] is None:
            try:
                result["raw_content"] = load_web_page(result["url"])
            except Exception as e:
                print(f"Error loading page: {result['url']}")
                print(e)
                result["raw_content"] = result["content"]

    resources_json_path = (
        f"{current_path}/data/resources_"
        f"{datetime.now().strftime('%Y_%m%d_%H%M%S')}.json"
    )

    with open(resources_json_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

    return results, resources_json_path

### web_page_to_document() — 검색 결과를 Document로 변환
웹 검색 결과 딕셔너리를 LangChain의 `Document` 객체로 변환합니다.
검색 결과의 본문을 `page_content`로 저장하고 웹페이지 제목과 URL을 `metadata`에 추가합니다.

**매개변수**
| 매개변수 | 타입 | 설명 |
|---|---|---|
| `web_page` | `dict` | Tavily 웹 검색 결과를 담은 딕셔너리 |

**사용하는 딕셔너리 필드**
| 필드 | 설명 |
|---|---|
| `raw_content` | 웹페이지에서 수집한 원본 콘텐츠 |
| `content` | Tavily가 제공한 콘텐츠 |
| `title` | 웹페이지 제목 |
| `url` | 웹페이지 URL |

**반환값**
- `Document`: LangChain 문서 객체

**주요 처리 과정**
1. `raw_content`와 `content`의 문자열 길이를 비교합니다.
2. 더 긴 콘텐츠를 `page_content`로 선택합니다.
3. 웹페이지 제목과 URL을 `metadata`에 저장합니다.
4. `Document` 객체를 생성하여 반환합니다.

In [5]:
from langchain_core.documents import Document

def web_page_to_document(web_page):
    # raw_content와 content 중 정보가 많은 것을 page_content로 사용
    if len(web_page["raw_content"]) > len(web_page["content"]):
        page_content = web_page["raw_content"]
    else:
        page_content = web_page["content"]

    document = Document(
        page_content=page_content,
        metadata={
            "title": web_page["title"],
            "source": web_page["url"]
        }
    )

    return document

### web_page_json_to_documents() — JSON 파일을 Document 목록으로 변환

`web_search()`에서 저장한 JSON 파일을 읽어 웹 검색 결과를 LangChain의 `Document` 객체 목록으로 변환합니다.  
각 검색 결과는 `web_page_to_document()`를 통해 개별 문서로 생성됩니다.

**매개변수**
| 매개변수 | 타입 | 설명 |
|---|---|---|
| `json_file` | `str` | 검색 결과가 저장된 JSON 파일 경로 |

**반환값**
- `list[Document]`: 변환된 LangChain 문서 객체 목록

**주요 처리 과정**
1. 지정된 JSON 파일을 UTF-8 인코딩으로 읽습니다.
2. `json.load()`를 사용하여 JSON 데이터를 Python 객체로 변환합니다.
3. 각 웹페이지 검색 결과에 대해 `web_page_to_document()`를 호출합니다.
4. 생성된 문서를 리스트에 추가합니다.
5. 완성된 `Document` 목록을 반환합니다.

**참고**
이 함수는 JSON 파일을 읽어오는 기능과 문서 변환 기능을 연결하는 역할을 합니다. 실제 문서 객체의 생성은 `web_page_to_document()`가 담당합니다.

In [6]:
def web_page_json_to_documents(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        resources = json.load(f)

    documents = []

    for web_page in resources:
        document = web_page_to_document(web_page)
        documents.append(document)

    return documents

### split_documents() — 문서 청크 분할
LangChain의 `RecursiveCharacterTextSplitter`를 사용하여 긴 문서를 일정한 크기의 청크(Chunk)로 분할합니다.
분할된 문서는 벡터 데이터베이스에 저장되며, 이후 RAG 검색에서 관련 콘텐츠를 찾는 데 사용됩니다.

**매개변수**
| 매개변수 | 타입 | 기본값 | 설명 |
|---|---|---|---|
| `documents` | `list[Document]` | 필수 | 분할할 Document 목록 |
| `chunk_size` | `int` | `1000` | 청크의 최대 크기 |
| `chunk_overlap` | `int` | `100` | 인접한 청크 사이에 중첩할 크기 |

**반환값**
- `list[Document]`: 청크 단위로 분할된 문서 목록

**주요 처리 과정**
1. 입력된 문서의 개수를 확인합니다.
2. `RecursiveCharacterTextSplitter`를 생성합니다.
3. `chunk_size`, `chunk_overlap`을 설정합니다.
4. `split_documents()`로 문서를 청크 단위로 분할합니다.
5. 생성된 청크의 개수를 출력하고 결과를 반환합니다.

**RecursiveCharacterTextSplitter**
텍스트의 구조를 최대한 유지하면서 분할하는 LangChain의 텍스트 분할기입니다.  
기본적으로 문단, 줄바꿈, 공백 등의 구분자를 우선적으로 고려하여 텍스트를 분할합니다.

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=1000, chunk_overlap=100):
    print("Splitting documents...")
    print(
        f"{len(documents)}개의 문서를 {chunk_size}자 크기로 "
        f"중첩 {chunk_overlap}자로 분할합니다.\n"
    )

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    splits = text_splitter.split_documents(documents)

    print(f"총 {len(splits)}개의 문서로 분할되었습니다.")
    return splits

### documents_to_chroma() — Chroma 벡터 DB 저장
LangChain Document 목록을 Chroma 벡터 데이터베이스에 저장합니다.  
기존 데이터베이스에 저장된 URL과 새로운 문서의 URL을 비교하여, 저장되지 않은 URL에 해당하는 문서만 추가합니다.

**매개변수**
| 매개변수 | 타입 | 기본값 | 설명 |
|---|---|---|---|
| `documents` | `list[Document]` | 필수 | Chroma에 저장할 문서 목록 |
| `chunk_size` | `int` | `1000` | 문서 분할 시 사용할 청크 크기 |
| `chunk_overlap` | `int` | `100` | 청크 사이에 중첩할 크기 |

**반환값**
- `None`: 별도의 값을 반환하지 않습니다.

**주요 처리 과정**
1. 입력 문서의 메타데이터에서 URL을 추출합니다.
2. Chroma DB에 이미 저장된 문서의 URL을 조회합니다.
3. 두 URL 집합을 비교하여 새로운 URL을 구합니다.
4. 새로운 URL에 해당하는 문서만 필터링합니다.
5. `split_documents()`를 이용하여 문서를 청크 단위로 분할합니다.
6. `vectorstore.add_documents()`를 호출하여 Chroma DB에 저장합니다.

**주요 객체 및 메서드**
| 항목 | 설명 |
|---|---|
| `vectorstore._collection.get()` | Chroma 내부 Collection에서 저장된 데이터를 조회 |
| `set()` | 중복을 제거한 집합 생성 |
| `split_documents()` | 문서를 청크 단위로 분할 |
| `vectorstore.add_documents()` | 문서를 임베딩한 후 Chroma에 저장 |

In [8]:
def documents_to_chroma(documents, chunk_size=1000, chunk_overlap=100):
    print("Documents를 Chroma DB에 저장합니다.")

    # documents의 URL 가져오기
    urls = [document.metadata["source"] for document in documents]

    # 이미 vectorstore에 저장된 URL 가져오기
    stored_metadatas = vectorstore._collection.get()["metadatas"]
    stored_web_urls = [
        metadata["source"]
        for metadata in stored_metadatas
    ]

    # 새로운 URL만 남기기
    new_urls = set(urls) - set(stored_web_urls)

    new_documents = []

    for document in documents:
        if document.metadata["source"] in new_urls:
            new_documents.append(document)
            print(document.metadata)

    # 새로운 문서 분할
    splits = split_documents(
        new_documents,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    # Chroma DB에 저장
    if splits:
        vectorstore.add_documents(splits)
    else:
        print("No new urls to process")

### add_web_pages_json_to_chroma() — JSON 문서의 Chroma 저장 자동화
JSON 파일에 저장된 웹 검색 결과를 읽고, LangChain Document 객체로 변환한 후 Chroma 벡터 데이터베이스에 저장합니다.  
`web_page_json_to_documents()`와 `documents_to_chroma()`를 연결하여 문서 저장 과정을 한 번에 처리하는 함수입니다.

**매개변수**
| 매개변수 | 타입 | 기본값 | 설명 |
|---|---|---|---|
| `json_file` | `str` | 필수 | 웹 검색 결과 JSON 파일 경로 |
| `chunk_size` | `int` | `1000` | 문서 분할 시 사용할 청크 크기 |
| `chunk_overlap` | `int` | `100` | 청크 사이에 중첩할 크기 |

**반환값**
- `None`: 별도의 값을 반환하지 않습니다.

**주요 처리 과정**
1. `web_page_json_to_documents()`로 JSON 파일을 Document 목록으로 변환합니다.
2. 변환된 문서를 `documents_to_chroma()`에 전달합니다.
3. 신규 URL 확인, 청크 분할, 임베딩 생성, Chroma 저장 과정을 수행합니다.

In [9]:
def add_web_pages_json_to_chroma(
    json_file,
    chunk_size=1000,
    chunk_overlap=100
):
    documents = web_page_json_to_documents(json_file)

    documents_to_chroma(
        documents,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

### retrieve() — Chroma 벡터 검색 Tool
사용자가 입력한 검색어를 기반으로 Chroma 벡터 데이터베이스에서 의미적으로 유사한 문서를 검색합니다.  
검색어는 OpenAI Embeddings를 통해 벡터로 변환되며, 저장된 문서 벡터와의 유사도를 기반으로 관련 문서를 반환합니다.  
`@tool` 데코레이터를 사용하여 LangChain 에이전트가 호출할 수 있는 검색 도구로 정의됩니다.

**매개변수**
| 매개변수 | 타입 | 기본값 | 설명 |
|---|---|---|---|
| `query` | `str` | 필수 | 유사한 문서를 검색할 검색어 |
| `top_k` | `int` | `5` | 최대 검색 문서 수 |

**반환값**
- `list[Document]`: 검색어와 유사한 문서 목록

**주요 처리 과정**
1. `vectorstore.as_retriever()`로 검색기(Retriever)를 생성합니다.
2. `search_kwargs`에 최대 검색 문서 수를 설정합니다.
3. `retriever.invoke(query)`로 유사도 검색을 실행합니다.
4. 검색된 Document 목록을 반환합니다.

In [10]:
@tool
def retrieve(query: str, top_k: int = 5):
    """
    주어진 query에 대해 벡터 검색을 수행하고, 결과를 반환한다.
    """
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": top_k}
    )

    retrieved_docs = retriever.invoke(query)

    return retrieved_docs